In [ ]:
# ============================================================
# CELDA S1
# ============================================================
import pandas as pd

import os
REPO_URL = "https://github.com/Camilamop/RENABAP.git"

if os.path.exists('/content/RENABAP'):
    !cd /content/RENABAP && git fetch origin && git reset --hard origin/master
else:
    !git clone {REPO_URL} /content/RENABAP

!pip install geopandas --quiet

path_2023 = "/content/RENABAP/data/raw/RENABAP_2023.csv"
path_2022 = "/content/RENABAP/data/raw/RENABAP_2022.csv"
path_2018 = "/content/RENABAP/data/raw/RENABAP_2018.csv"
path_diccionario_2018 = "/content/RENABAP/data/raw/RENABAP_2018_diccionario.csv"

raw_2023 = pd.read_csv(path_2023, dtype=str, low_memory=False)
raw_2022 = pd.read_csv(path_2022, dtype=str, low_memory=False)
raw_2018 = pd.read_csv(path_2018, dtype=str, low_memory=False)

print("2023:", raw_2023.shape, "| 2022:", raw_2022.shape, "| 2018:", raw_2018.shape)

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 3), reused 8 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 11.35 KiB | 1.89 MiB/s, done.
From https://github.com/Camilamop/RENABAP
   e686ec5..da39965  master     -> origin/master
HEAD is now at da39965 diccionario
2023: (6467, 19) | 2022: (5687, 19) | 2018: (4416, 30)


In [ ]:
# ============================================================
# CELDA S2B — Parseo del diccionario de datos 2018
# Formato del archivo: bloques "Variable:" / "Etiqueta:" / tabla "Valores"-"Etiquetas"
# ============================================================
def parse_diccionario_renabap(path):
    df = pd.read_csv(path, header=None, dtype=str)
    variables = {}
    var_actual = None
    en_tabla_valores = False
    for _, row in df.iterrows():
        c0, c1 = row[0], row[1]
        if pd.isna(c0) and pd.isna(c1):
            en_tabla_valores = False
            continue
        if c0 == "Variable:":
            var_actual = c1.strip()
            variables[var_actual] = {"etiqueta_columna": None, "valores": {}}
            en_tabla_valores = False
        elif c0 == "Etiqueta:":
            if var_actual:
                variables[var_actual]["etiqueta_columna"] = c1
        elif c0 == "Valores":
            en_tabla_valores = True
        elif en_tabla_valores and var_actual:
            try:
                variables[var_actual]["valores"][int(c0)] = c1
            except (ValueError, TypeError):
                pass
    return variables

DICCIONARIO_2018 = parse_diccionario_renabap(path_diccionario_2018)

COLS_RAW_2018 = {
    "electricidad": "Electricidad",
    "cloacas": "Disposición de excretas",
    "agua": "Acceso al agua",
    "cocina": "Energía para cocinar",
    "calefaccion": "Energía para calefaccionar",
}
VAR_A_ESQUEMA = {
    "electricidad": "energia_electrica",
    "cloacas": "efluentes_cloacales",
    "agua": "agua_corriente",
    "cocina": "cocina",
    "calefaccion": "calefaccion",
}


for var, col in COLS_RAW_2018.items():
    assert DICCIONARIO_2018[var]["etiqueta_columna"] == col, f"Desajuste en {var}"
print("Diccionario 2018 parseado y verificado ✓")

def decodificar_2018(raw_df, diccionario, cols_raw, var_a_esquema):
    out = pd.DataFrame({"id_renabap": raw_df["id_renabap"]})
    for var, col_raw in cols_raw.items():
        nombre_esquema = var_a_esquema[var]
        codigo_num = pd.to_numeric(raw_df[col_raw], errors="coerce")
        out[nombre_esquema] = codigo_num.map(diccionario[var]["valores"])
    out["anio_renabap"] = 2018
    return out[["id_renabap", "energia_electrica", "efluentes_cloacales",
                "agua_corriente", "cocina", "calefaccion", "anio_renabap"]]

serv_2018 = decodificar_2018(raw_2018, DICCIONARIO_2018, COLS_RAW_2018, VAR_A_ESQUEMA)

Diccionario 2018 parseado y verificado ✓


In [ ]:
# ============================================================
# CELDA S2 — Normalización de nombres de columnas de servicios

# ============================================================
MAPA_COLUMNAS = {
    2023: {
        "id_renabap": "id_renabap",
        "energia_electrica": "energia_electrica",
        "efluentes_cloacales": "efluentes_cloacales",
        "agua_corriente": "agua_corriente",
        "cocina": "cocina",
        "calefaccion": "calefaccion",
    },
    2022: {
        "id_renabap": "renabap_id",   # ojo: nombre distinto en 2022
        "energia_electrica": "energia_electrica",
        "efluentes_cloacales": "efluentes_cloacales",
        "agua_corriente": "agua_corriente",
        "cocina": "cocina",
        "calefaccion": "calefaccion",
    },
}

def normalizar_servicios(df, anio):
    m = MAPA_COLUMNAS[anio]
    out = df[[m["id_renabap"], m["energia_electrica"], m["efluentes_cloacales"],
              m["agua_corriente"], m["cocina"], m["calefaccion"]]].copy()
    out.columns = ["id_renabap", "energia_electrica", "efluentes_cloacales",
                   "agua_corriente", "cocina", "calefaccion"]
    out["anio_renabap"] = anio
    # "Sin Datos" es un no-dato disfrazado de string, no una categoría real
    out = out.replace("Sin Datos", pd.NA)
    return out

serv_2023 = normalizar_servicios(raw_2023, 2023)
serv_2022 = normalizar_servicios(raw_2022, 2022)
servicios_largo = pd.concat([serv_2018, serv_2022, serv_2023], ignore_index=True)
print(servicios_largo.groupby("anio_renabap").size())

anio_renabap
2018    4416
2022    5687
2023    6467
dtype: int64


In [ ]:
# ============================================================
# CELDA S3 — Cantidad de datos y faltantes por servicio y año
# ============================================================
cols_servicio = ["energia_electrica", "efluentes_cloacales", "agua_corriente",
                  "cocina", "calefaccion"]

resumen_completitud = []
for anio, grupo in servicios_largo.groupby("anio_renabap"):
    n_total = len(grupo)
    for col in cols_servicio:
        n_validos = grupo[col].notna().sum()
        resumen_completitud.append({
            "anio": anio,
            "servicio": col,
            "n_total": n_total,
            "n_validos": n_validos,
            "n_faltantes": n_total - n_validos,
            "pct_faltante": round((n_total - n_validos) / n_total * 100, 2),
        })

df_completitud = pd.DataFrame(resumen_completitud)
df_completitud

,anio,servicio,n_total,n_validos,n_faltantes,pct_faltante
0,2018,energia_electrica,4416,4416,0,0.00
1,2018,efluentes_cloacales,4416,4416,0,0.00
2,2018,agua_corriente,4416,4416,0,0.00
3,2018,cocina,4416,4416,0,0.00
4,2018,calefaccion,4416,4416,0,0.00
5,2022,energia_electrica,5687,5687,0,0.00
6,2022,efluentes_cloacales,5687,5687,0,0.00
7,2022,agua_corriente,5687,5687,0,0.00
8,2022,cocina,5687,5687,0,0.00
9,2022,calefaccion,5687,5041,646,11.36


In [ ]:
# ============================================================
# CELDA S4 — Valores únicos por servicio y año
# ============================================================
valores_unicos = []
for anio, grupo in servicios_largo.groupby("anio_renabap"):
    for col in cols_servicio:
        vc = grupo[col].value_counts(dropna=True)
        for valor, n in vc.items():
            valores_unicos.append({"anio": anio, "servicio": col, "valor": valor, "n": n})

df_valores_unicos = pd.DataFrame(valores_unicos).sort_values(["servicio", "anio", "n"], ascending=[True, True, False])
df_valores_unicos

,anio,servicio,valor,n
17,2018,agua_corriente,Conexión irregular a la red pública de agua co...,2619
18,2018,agua_corriente,Bomba de agua de pozo domiciliaria,692
19,2018,agua_corriente,Bomba de agua de pozo comunitaria,243
20,2018,agua_corriente,Conexión formal al agua corriente de red pública,235
21,2018,agua_corriente,Sin conexión formal al agua corriente de red p...,141
...,...,...,...,...
85,2023,energia_electrica,Conexión a la red con medidor compartido,85
86,2023,energia_electrica,Conexión regular a la red con medidor domicili...,53
87,2023,energia_electrica,Conexión regular a la red con medidor prepago,44
88,2023,energia_electrica,Generador eléctrico a combustión,7


In [ ]:
# ============================================================
# CELDA S5 — Categorización tripartita: formal / informal / inexistente-precario
#
# CRITERIO (editable): -> "formal"/"regular" = conexión legítima a la red (con o sin
# factura); "irregular"/"comunitario"/"compartido" = conectado a la red pero
# de manera no regularizada/no individualizada; todo lo que queda FUERA de
# la red (pozo, cisterna, garrafa, leña, generador, etc.) = inexistente/precario.
#
# ⚠️ DOS CASOS QUE TE DEJO MARCADOS PARA QUE DECIDAS (ver pregunta al final):
#   - Agua de pozo domiciliaria / cámara séptica: hoy están en "inexistente/
#     precario" porque están fuera de la red formal, pero en la literatura de
#     déficit habitacional (INDEC/NBI) a veces se consideran soluciones
#     "adecuadas" aunque no sean de red. Si querés tratarlas como una cuarta
#     categoría o subirlas a "informal", cambiá el diccionario abajo.
#   - "Energía eléctrica" para cocinar/calefaccionar: hoy en "inexistente/
#     precario" (fuera de la red de gas), aunque no es intrínsecamente
#     precaria si el suministro eléctrico es estable.
# ⚠️ - "Otro/Vacío" (código más alto de cada servicio) -> queda sin categoría (NA),
#     igual tratamiento que "Sin Datos" en 2022/2023.
#   - Agua "Cobertura Parcial (RPPVAP)" -> hoy en "informal" (hay conexión a la
#     red pero no está resuelta/regularizada).
#   - Agua "Sin conexión formal ... (RPPVAP)" -> hoy en "inexistente_precario"
#     (el propio rótulo lo dice). Estas dos categorías no existen en 2022/2023;
# ============================================================

CATEGORIZACION = {
    "energia_electrica": {
        "formal": [
            "Conexión formal a la red con medidor domiciliario con factura",
            "Conexión regular a la red con medidor domiciliario pero sin factura",
            "Conexión regular a la red con medidor domiciliario con consumo limitado",
            "Conexión regular a la red con medidor prepago",
        ],
        "informal": [
            "Conexión irregular a la red",
            "Conexión a la red con medidor comunitario",
            "Conexión a la red con medidor compartido",
            "Energía solar",
        ],
        "inexistente_precario": [
            "No tiene conexión eléctrica",
            "Generador eléctrico a combustión",
        ],
    },
    "agua_corriente": {
        "formal": [
            "Conexión formal a la red de agua con factura",
            "Conexión regular a la red de agua pero sin factura",
        ],
        "informal": [
            "Conexión irregular a la red de agua",
            "Canilla comunitaria dentro del barrio",
        ],
        "inexistente_precario": [
            "Bomba de agua de pozo domiciliaria",
            "Bomba de agua de pozo comunitaria",
            "Camión cisterna",
            "Acarreo de baldes/recipientes desde fuera del barrio",
            "Vertiente, arroyo, río o canal",
            "Cosecha/recolección de agua de lluvia",
        ],
    },
    "efluentes_cloacales": {
        "formal": [
            "Conexión formal a la red cloacal",
        ],
        "informal": [
            "Conexión irregular a la red cloacal",
            "Red cloacal conectada a la red pluvial",
        ],
        "inexistente_precario": [
            "Desagüe a cámara séptica y pozo ciego",
            "Desagüe sólo a pozo negro/ciego u hoyo",
            "Desagüe a intemperie o cuerpo de agua",
            "Baño seco",
            "Biodigestor para tratar efluentes cloacales",
        ],
    },
    "cocina": {
        "formal": [
            "Conexión formal a la red de gas con factura",
            "Energía eléctrica",
        ],
        "informal": [
            "Conexión irregular a la red de gas",
        ],
        "inexistente_precario": [
            "Gas en garrafa",
            "Leña o carbón",
        ],
    },
    "calefaccion": {
        "formal": [
            "Conexión formal a la red de gas con factura",
            "Energía eléctrica",
        ],
        "informal": [
            "Conexión irregular a la red de gas",
        ],
        "inexistente_precario": [
            "Gas en garrafa",
            "Leña o carbón",
            "Inexistente",
        ],
    },
}

def construir_lookup(categorizacion):
    lookup = {}
    for servicio, categorias in categorizacion.items():
        for cat, valores in categorias.items():
            for v in valores:
                lookup[(servicio, v)] = cat
    return lookup

LOOKUP = construir_lookup(CATEGORIZACION)

def categorizar(df, lookup, cols):
    out = df.copy()
    for col in cols:
        out[f"{col}_cat"] = out[col].apply(
            lambda v: lookup.get((col, v), pd.NA) if pd.notna(v) else pd.NA
        )
    return out

servicios_categorizado = categorizar(servicios_largo, LOOKUP, cols_servicio)

CATEGORIZACION_2018 = {
    "energia_electrica": {
        "formal": [
            "Conexión regular a la red pública con medidores domiciliarios pero sin boleta/factura",
            "Conexión formal a la red pública de energía eléctrica con medidores domiciliarios.",
            "Acceso formal con consumo limitado",
            "Energía solar",
        ],
        "informal": [
            "Conexión a la red pública con medidor comunitario / social",
            "Conexión irregular a la red pública",
        ],
        "inexistente_precario": [
            "Generador eléctrico a combustión", "No tiene conexión eléctrica",
        ],
    },
    "efluentes_cloacales": {
        "formal": ["Conexión formal a la red cloacal pública"],
        "informal": ["Red cloacal conectada al pluvial", "Conexión irregular a la red cloacal pública"],
        "inexistente_precario": [
            "Desagüe a cámara séptica y pozo ciego", "Desagüe sólo a pozo negro/ciego u hoyo",
            "Desagüe a intemperie o cuerpo de agua", "Baño seco",
        ],
    },
    "agua_corriente": {
        "formal": [
            "Conexión formal al agua corriente de red pública",
            "Conexión regular al agua corriente de red pública pero sin boleta/factura",
        ],
        "informal": [
            "Conexión irregular a la red pública de agua corriente",
            "Canilla comunitaria dentro del barrio",
            "Cobertura Parcial (RPPVAP)",
        ],
        "inexistente_precario": [
            "Bomba de agua de pozo domiciliaria", "Bomba de agua de pozo comunitaria", "Camión cisterna",
            "Acarreo de baldes y/o bidones desde fuera del barrio",
            "Sin conexión formal al agua corriente de red pública (RPPVAP)",
            "Vertiente, arroyo, río o canal",
        ],
    },
    "cocina": {
        "formal": ["Conexión formal al gas natural de red pública","Energía eléctrica"],
        "informal": ["Conexión irregular a la red de gas natural"],
        "inexistente_precario": [ "Gas en garrafa", "Leña o carbón"],
    },
    "calefaccion": {
        "formal": ["Conexión formal al gas natural de red pública","Energía eléctrica"],
        "informal": ["Conexión irregular a la red de gas natural"],
        "inexistente_precario": [ "Gas en garrafa", "Leña o carbón", "Inexistente"],
    },
}

LOOKUP = construir_lookup(CATEGORIZACION)
LOOKUP.update(construir_lookup(CATEGORIZACION_2018))

# chequeo de cobertura: valores que no matchean ningún diccionario
# (deberían ser 0 filas si el diccionario está completo)
for col in cols_servicio:
    no_mapeados = servicios_categorizado[
        servicios_categorizado[col].notna() & servicios_categorizado[f"{col}_cat"].isna()
    ][col].unique()
    if len(no_mapeados) > 0:
        print(f"⚠️ {col}: valores sin categorizar -> {no_mapeados}")

servicios_categorizado.head()

⚠️ energia_electrica: valores sin categorizar -> ['Conexión irregular a la red pública'
 'Conexión formal a la red pública de energía eléctrica con medidores domiciliarios.'
 'Conexión a la red pública con medidor comunitario / social'
 'Otro / vacío' 'Acceso formal con consumo limitado'
 'Conexión regular a la red pública con medidores domiciliarios pero sin boleta/factura']
⚠️ efluentes_cloacales: valores sin categorizar -> ['Conexión formal a la red cloacal pública'
 'Conexión irregular a la red cloacal pública'
 'Red cloacal conectada al pluvial' 'Otro / vacío']
⚠️ agua_corriente: valores sin categorizar -> ['Conexión irregular a la red pública de agua corriente'
 'Conexión formal al agua corriente de red pública'
 'Sin conexión formal al agua corriente de red pública (RPPVAP)'
 'Cobertura Parcial (RPPVAP)' 'Otro / Vacío'
 'Conexión regular al agua corriente de red pública pero sin boleta/factura'
 'Acarreo de baldes y/o bidones desde fuera del barrio']
⚠️ cocina: valores sin categ

,id_renabap,energia_electrica,efluentes_cloacales,agua_corriente,cocina,calefaccion,anio_renabap,energia_electrica_cat,efluentes_cloacales_cat,agua_corriente_cat,cocina_cat,calefaccion_cat
0,1,Conexión irregular a la red pública,Desagüe sólo a pozo negro/ciego u hoyo,Conexión irregular a la red pública de agua co...,Gas en garrafa,Otro / Vacío,2018,<NA>,inexistente_precario,<NA>,inexistente_precario,<NA>
1,2,Conexión irregular a la red pública,Desagüe a cámara séptica y pozo ciego,Conexión irregular a la red pública de agua co...,Gas en garrafa,Leña o carbón,2018,<NA>,inexistente_precario,<NA>,inexistente_precario,inexistente_precario
2,3,Conexión irregular a la red pública,Desagüe sólo a pozo negro/ciego u hoyo,Conexión formal al agua corriente de red pública,Gas en garrafa,Leña o carbón,2018,<NA>,inexistente_precario,<NA>,inexistente_precario,inexistente_precario
3,4,Conexión irregular a la red pública,Desagüe sólo a pozo negro/ciego u hoyo,Conexión irregular a la red pública de agua co...,Gas en garrafa,Energía eléctrica,2018,<NA>,inexistente_precario,<NA>,inexistente_precario,formal
4,5,Conexión irregular a la red pública,Desagüe sólo a pozo negro/ciego u hoyo,Sin conexión formal al agua corriente de red p...,Gas en garrafa,Otro / Vacío,2018,<NA>,inexistente_precario,<NA>,inexistente_precario,<NA>


In [ ]:
# ============================================================
# CELDA S6 (BONUS) — Índice de precariedad de servicios por barrio
# Cuenta cuántos de los 5 servicios están en "inexistente_precario"
# Útil para el análisis 1 (caracterización del universo) y para
# comparar barrios con y sin intervención (análisis 5)
# ============================================================
cat_cols = [f"{c}_cat" for c in cols_servicio]

servicios_categorizado["n_servicios_precarios"] = (
    servicios_categorizado[cat_cols] == "inexistente_precario"
).sum(axis=1)

servicios_categorizado["n_servicios_validos"] = servicios_categorizado[cat_cols].notna().sum(axis=1)

resumen_indice = servicios_categorizado.groupby("anio_renabap")["n_servicios_precarios"].describe()
resumen_indice

,count,mean,std,min,25%,50%,75%,max
anio_renabap,,,,,,,,
2018,4416.0,2.659194,0.783477,0.0,2.0,3.0,3.0,5.0
2022,5687.0,2.789344,0.802699,0.0,2.0,3.0,3.0,5.0
2023,6467.0,2.820782,0.797584,0.0,2.0,3.0,3.0,5.0


In [ ]:
# ============================================================
# CELDA S8 (BONUS) — Comparación temporal 2022 → 2023 por barrio
# Directamente relevante para la hipótesis: ¿mejoró, empeoró o se
# mantuvo el acceso a cada servicio en los barrios con intervención?
# ============================================================
comparacion = servicios_categorizado[servicios_categorizado["anio_renabap"].isin([2022, 2023])]

pivot_list = []
for col in cols_servicio:
    piv = comparacion.pivot(index="id_renabap", columns="anio_renabap", values=f"{col}_cat")
    piv.columns = [f"{col}_2022", f"{col}_2023"]
    pivot_list.append(piv)

comparacion_wide = pd.concat(pivot_list, axis=1).reset_index()

for col in cols_servicio:
    c22, c23 = f"{col}_2022", f"{col}_2023"
    orden = {"inexistente_precario": 0, "informal": 1, "formal": 2}
    comparacion_wide[f"{col}_cambio"] = comparacion_wide.apply(
        lambda r: "mejoró" if pd.notna(r[c22]) and pd.notna(r[c23]) and orden.get(r[c23], -1) > orden.get(r[c22], -1)
        else ("empeoró" if pd.notna(r[c22]) and pd.notna(r[c23]) and orden.get(r[c23], -1) < orden.get(r[c22], -1)
        else ("igual" if pd.notna(r[c22]) and pd.notna(r[c23]) else pd.NA)),
        axis=1
    )

for col in cols_servicio:
    print(f"\n{col}:")
    print(comparacion_wide[f"{col}_cambio"].value_counts(dropna=False))


energia_electrica:
energia_electrica_cambio
igual    5654
<NA>      846
Name: count, dtype: int64

efluentes_cloacales:
efluentes_cloacales_cambio
igual    5654
<NA>      846
Name: count, dtype: int64

agua_corriente:
agua_corriente_cambio
igual    5654
<NA>      846
Name: count, dtype: int64

cocina:
cocina_cambio
igual     5653
<NA>       846
mejoró       1
Name: count, dtype: int64

calefaccion:
calefaccion_cambio
igual    5010
<NA>     1490
Name: count, dtype: int64
